In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
from pypdf  import PdfReader

"Create README, Create venv using .toml file, add /data to .gitignore". 

In [3]:
# Read PDF and extract text

path_input="data/Code général des impôts - CH1 - 1-373.pdf"

pdf_reader = PdfReader(path_input)
text_content = ""
for page in pdf_reader.pages:
    text_content += page.extract_text()

In [4]:

# Chunk the data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_text(text_content)

print(f"Total pages: {len(pdf_reader.pages)}")
print(f"Total text length: {len(text_content)} characters")
print(f"Number of chunks: {len(chunks)}\n")

print("First chunk preview:\n\n"+f"{chunks[0]}\n")
print("Chunk No 10 preview:\n\n"+f"{chunks[9]}\n")
print("Chunk No 100 preview:\n\n"+f"{chunks[99]}\n")

Total pages: 373
Total text length: 1803712 characters
Number of chunks: 2220

First chunk preview:

Code général des impôts
Dernière modification: 2026-03-14
Edition : 2026-03-14
2419 articles avec 5773 liens
818 références externes
Ce code ne contient que du droit positif français,
les articles et éléments abrogés ne sont pas inclus.
Il est recalculé au fur et à mesure des mises à jour.
Pensez à actualiser votre copie régulièrement à partir de codes.droit.org.
Ces codes ont pour objectif de démontrer l’utilité de l’ouverture des données publiques juridiques tant législatives que
jurisprudentielles. Il s’y ajoute une promotion du mouvement Open Science Juridique avec une incitation au dépôt du
texte intégral en accès ouvert des articles de doctrine venant du monde professionnel (Grande Bibliothèque du Droit) et
universitaire (HAL-CNRS).
Traitements effectués à partir des données issues des APIs Legifrance et Judilibre. droit.org remercie les acteurs du
Web qui autorisent des liens ver

In [5]:
from sentence_transformers import SentenceTransformer

In [6]:


# Chargement du modèle (téléchargé automatiquement la première fois)
modele = SentenceTransformer('all-MiniLM-L6-v2')

# Vectorisation de tous les chunks d'un coup
vecteurs = modele.encode(chunks, show_progress_bar=True)

print(f"Nombre de chunks vectorisés : {len(vecteurs)}")
print(f"Dimensions de chaque vecteur : {vecteurs.shape[1]}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/70 [00:00<?, ?it/s]

Nombre de chunks vectorisés : 2220
Dimensions de chaque vecteur : 384


In [10]:
print(type(vecteurs))
print(len(vecteurs))
print(type(vecteurs[0]))
print(len(vecteurs[0]))

<class 'numpy.ndarray'>
2220
<class 'numpy.ndarray'>
384


In [11]:
import chromadb

# Connexion à ChromaDB en mode persistant (sauvegarde sur disque)
client = chromadb.PersistentClient(path="./ma_bdd_juridique")

# Création de la collection (comme une table en SQL)
collection = client.get_or_create_collection(
    name="code_civil",
    metadata={"hnsw:space": "cosine"}  # on utilise la similarité cosinus
)

In [12]:
# Stockage des chunks + vecteurs + métadonnées
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=vecteurs.tolist(),
    metadatas=[
        {
            "source": "Code civil",
            "chunk_index": i
        }
        for i in range(len(chunks))
    ]
)

print(f"✅ {collection.count()} chunks stockés dans ChromaDB")

✅ 2220 chunks stockés dans ChromaDB


In [13]:
resultats = collection.get(
    ids=["chunk_0", "chunk_1"],
    include=["documents", "metadatas"]
)

for doc, meta in zip(resultats["documents"], resultats["metadatas"]):
    print(f"Source : {meta['source']}")
    print(f"Texte  : {doc[:100]}...")
    print("---")

Source : Code civil
Texte  : Code général des impôts
Dernière modification: 2026-03-14
Edition : 2026-03-14
2419 articles avec 57...
---
Source : Code civil
Texte  : Web qui autorisent des liens vers leur production : Dictionnaire du Droit Privé  (réalisé par MM. Se...
---
